In [1]:
import pandas as pd
import seaborn as sns

In [2]:
VIDGEN_DATASET_PATH = "/kaggle/input/cyberbullying/raw/Dynamically Generated Hate Dataset v0.2.2.csv"
def load_vidgen_dataset(path=VIDGEN_DATASET_PATH):
    """Loads the Vidgen dataset and maps labels."""
    vidgen_df = pd.read_csv(path)
    # Ensure the column name is correct, based on your notebook it was missing but used later.
    # Assuming 'label' is the column that needs mapping.
    vidgen_df = vidgen_df[['text', 'label']] # Adjusted based on your usage
    label_map = {'nothate': 0, 'hate': 1}
    vidgen_df['label'] = vidgen_df['label'].map(label_map)
    print(f"Loaded Vidgen dataset from {path}")
    print("Vidgen Label distribution:\n", vidgen_df['label'].value_counts()) #
    return vidgen_df


In [3]:

import re
import emoji
import html

def clean_text(text):
    # 1. Decode HTML entities
    text = html.unescape(text)
    
    # 2. Lowercase the text
    text = text.lower()
    
    # 3. Replace URLs with a special token
    text = re.sub(r'http\S+|www.\S+', ' url ', text)
    
    # 4. Replace mentions and hashtags
    text = re.sub(r'@\w+', ' user ', text)
    text = re.sub(r'#\w+', ' hashtag ', text)
    
    # 5. Convert emojis to text (e.g., 😊 -> :smiling_face_with_smiling_eyes:)
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 6. Remove digits and punctuation (keep only words and spaces)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 7. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

import re
import spacy

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

def remove_repeated_words(text):
    # Replace 2 or more repeated words (consecutive) with a single word
    # e.g. "help help help" -> "help"
    return re.sub(r'\b(\w+)( \1\b)+', r'\1', text)

def remove_repeated_sentences(text):
    doc = nlp(text)
    seen = set()
    unique_sents = []
    for sent in doc.sents:
        sent_text = sent.text.strip()
        if sent_text not in seen:
            unique_sents.append(sent_text)
            seen.add(sent_text)
    return " ".join(unique_sents)

def clean_repeated(text):
    text = remove_repeated_words(text)
    text = remove_repeated_sentences(text)
    return text
def full_preprocess(text):
    text = clean_text(text)
    text = clean_repeated(text)
    return text

In [4]:
from transformers import RobertaModel, RobertaPreTrainedModel
import torch.nn as nn
import torch

class RobertaCNNForSequenceClassification(RobertaPreTrainedModel):
    def __init__(self, config, class_weights=None):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = RobertaModel(config)
        self.hidden_size = config.hidden_size
        self.convs = nn.ModuleList([nn.Conv1d(self.hidden_size, 100, k) for k in [2, 3, 4]])
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(300, self.num_labels)
        self.class_weights = class_weights
        self.init_weights()

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.roberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        x = x.permute(0, 2, 1)
        x = [torch.relu(conv(x)).max(dim=2)[0] for conv in self.convs]
        x = torch.cat(x, dim=1)
        x = self.dropout(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}


e:\Cyberbullying\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

VIDGEN_DATASET_PATH = r"E:\Cyberbullying\dataset\raw\Dynamically Generated Hate Dataset v0.2.2.csv"
def load_vidgen_dataset(path=VIDGEN_DATASET_PATH):
    """Loads the Vidgen dataset and maps labels."""
    vidgen_df = pd.read_csv(path)
    vidgen_df = vidgen_df[['text', 'label']] # Adjusted based on your usage
    label_map = {'nothate': 0, 'hate': 1}
    vidgen_df['label'] = vidgen_df['label'].map(label_map)
    print(f"Loaded Vidgen dataset from {path}")
    print("Vidgen Label distribution:\n", vidgen_df['label'].value_counts()) #
    return vidgen_df


In [8]:
from datasets import Dataset,ClassLabel
from transformers import RobertaTokenizer
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

df= load_vidgen_dataset()
df['text'] = df['text'].apply(clean_text)  # Apply the full preprocessing function
texts = df['text'].tolist()
labels = df['label'].tolist()  # Assuming 'label' is the column with binary labels

dataset = Dataset.from_dict({'text': texts, 'label': labels})
label_classes = ClassLabel(num_classes=2, names=["not_cyberbullying", "cyberbullying"])
dataset = dataset.cast_column("label", label_classes)

dataset = dataset.train_test_split(test_size=0.2, stratify_by_column='label')
train_labels = dataset['train']['label']
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
print("Class weights:", class_weights)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized_ds = dataset.map(tokenize, batched=True)
tokenized_ds = tokenized_ds.rename_column("label", "labels")
tokenized_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


Loaded Vidgen dataset from E:\Cyberbullying\dataset\raw\Dynamically Generated Hate Dataset v0.2.2.csv
Vidgen Label distribution:
 label
1    22262
0    18993
Name: count, dtype: int64


Casting the dataset: 100%|██████████| 41255/41255 [00:00<00:00, 569006.49 examples/s]


Class weights: [1.08608661 0.92655811]


Map: 100%|██████████| 8251/8251 [00:11<00:00, 693.73 examples/s]


In [9]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    }


In [13]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
)

model = RobertaCNNForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    class_weights=class_weights_tensor
)


e:\Cyberbullying\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of RobertaCNNForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initial

In [14]:
import matplotlib.pyplot as plt

def plot_train_history(trainer_state):
    logs = trainer_state.log_history
    epochs = []
    train_loss, val_loss = [], []
    val_acc, val_f1 = [], []

    for log in logs:
        if "loss" in log and "epoch" in log:
            epochs.append(log["epoch"])
            train_loss.append(log["loss"])
        if "eval_loss" in log:
            val_loss.append(log["eval_loss"])
            val_acc.append(log["eval_accuracy"])
            val_f1.append(log["eval_f1"])

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label="Train Loss")
    plt.plot(epochs[:len(val_loss)], val_loss, label="Val Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Loss Over Epochs")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs[:len(val_acc)], val_acc, label="Val Accuracy")
    plt.plot(epochs[:len(val_f1)], val_f1, label="Val F1")
    plt.xlabel("Epochs")
    plt.ylabel("Score")
    plt.title("Accuracy & F1 Over Epochs")
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
plot_train_history(trainer.state)
trainer.save_model("best_roberta_cnn_model")


C:\Users\user\AppData\Local\Temp\ipykernel_12956\356022495.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch

# Predict on validation set
predictions_output = trainer.predict(tokenized_ds['test'])
logits = predictions_output.predictions
labels = predictions_output.label_ids

# Convert to predicted class
predicted_probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
predicted_labels = np.argmax(predicted_probs, axis=1)


In [ ]:
def plot_confusion(y_true, y_pred, labels=["Not Cyberbullying", "Cyberbullying"]):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    plt.show()

plot_confusion(labels, predicted_labels)


In [ ]:
print("Classification Report:\n")
print(classification_report(labels, predicted_labels, target_names=["Not Cyberbullying", "Cyberbullying"]))

def plot_roc_curve(y_true, y_probs):
    fpr, tpr, _ = roc_curve(y_true, y_probs[:, 1])
    auc_score = roc_auc_score(y_true, y_probs[:, 1])

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc_score:.2f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_roc_curve(labels, predicted_probs)
